# Adversarial Training Loop — Notebook Runner

This notebook runs `src/train_loop.py` using the unified dataset generated by
`notebooks/setup.ipynb`.

It trains a classifier, generates adversarial variants with the mutator, and
iteratively hardens the model across rounds.


## 0. Install dependencies (Colab only)


In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q -r ../requirements.txt
    print('Colab: installed requirements.txt')
else:
    print('Local/Jupyter: using your existing environment')


## 1. Imports


In [ ]:
import sys
from pathlib import Path
import pandas as pd

# Make ../src importable
SRC_DIR = (Path.cwd().parent / 'src').resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from train_loop import TrainingConfig, AdversarialTrainer

print('SRC_DIR:', SRC_DIR)


## 2. Load unified dataset


In [ ]:
unified_path = Path('..') / 'data' / 'unified' / 'unified_dataset.csv'
if not unified_path.exists():
    raise FileNotFoundError(
        f"Missing {unified_path}. Run notebooks/setup.ipynb first to generate it."
    )

df = pd.read_csv(unified_path)
df = df.dropna(subset=['prompt']).copy()
df['prompt'] = df['prompt'].astype(str)
df['jailbreak'] = df['jailbreak'].astype(int)

print('Rows:', len(df))
print('Label distribution:')
print(df['jailbreak'].value_counts())


## 3. Build train/test split lists


In [ ]:
from sklearn.model_selection import train_test_split

texts = df['prompt'].tolist()
labels = df['jailbreak'].tolist()

x_train, x_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.15, stratify=labels, random_state=42
)

jailbreak_prompts = [t for t, y in zip(x_train, y_train) if y == 1]
benign_prompts = [t for t, y in zip(x_train, y_train) if y == 0]

print('Train jailbreak:', len(jailbreak_prompts))
print('Train benign   :', len(benign_prompts))
print('Test size      :', len(x_test))


## 4. Configure and run adversarial training


In [ ]:
# Start small for university Jupyter; scale up later
config = TrainingConfig(
    model_name='distilbert-base-uncased',
    num_rounds=2,
    num_epochs=2,
    batch_size=8,
    variants_per_seed=3,
    cache_threshold=0.72,
    output_dir='../checkpoints',
    cache_save_path='../jailbreak_cache',
    history_save_path='../results/training_history.json',
)

trainer = AdversarialTrainer(config)
final_model, final_cache = trainer.run(
    jailbreak_prompts=jailbreak_prompts,
    benign_prompts=benign_prompts,
    test_prompts=x_test,
    test_labels=y_test,
)

print('\nDone. Final cache size:', len(final_cache))


## 5. Load saved training history


In [ ]:
import json
history_path = Path('../results/training_history.json')
if history_path.exists():
    with open(history_path, 'r') as f:
        history = json.load(f)
    print('Rounds logged:', len(history))
    for row in history:
        print(row)
else:
    print('No history file found yet:', history_path)
